In [ ]:
import torch
import numpy as np
from collections import Counter


class EffectiveNumberWeights:
    """Effective Number of Samples 权重计算工具类 (基于 Cui et al., 2019)"""
    
    @staticmethod
    def compute_weights(class_counts, beta=0.9999, num_classes=None, return_type='torch', normalize=False):
        """
        计算 Effective Number 权重
        
        Args:
            class_counts: list/array/dict/Counter, 每个类别的样本数量
            beta: float, 超参数 (0, 1)，默认0.9999
            num_classes: int, 总类别数（可选）
            return_type: str, 返回类型 ('numpy', 'torch', 'dict')
            normalize: bool, 是否归一化权重
            
        Returns:
            根据return_type返回相应格式的权重
        """
        # 处理输入格式
        if isinstance(class_counts, (Counter, dict)):
            if num_classes is None:
                if not class_counts: # 处理空字典或空Counter
                    num_classes = 0
                else:
                    num_classes = max(class_counts.keys()) + 1
            counts_array = np.zeros(num_classes)
            for k, v in class_counts.items():
                if k < num_classes: # 确保键在范围内
                    counts_array[k] = v
        else:
            counts_array = np.array(class_counts)
            if num_classes is None:
                num_classes = len(counts_array)
        
        # 确保beta在有效范围内
        beta = max(0.0, min(0.9999, float(beta))) # 保持0.9999作为上限以避免beta=1
        
        weights = np.zeros(num_classes)
        
        # 计算每个类别的权重
        for i, count in enumerate(counts_array):
            if count > 0:
                # En = (1 - β^n) / (1 - β)
                if beta == 0:
                    effective_num = 1.0 # 当beta=0, E_n = 1
                else:
                    effective_num = (1.0 - beta**count) / (1.0 - beta)
                
                # 防止 effective_num 极小（比如 beta 极接近1，count又很小但非0时，1-beta**count 可能接近0）
                # 导致权重过大。但通常情况下，effective_num >= 1 (因为 beta**count <= 1)
                # 如果 effective_num 接近0 (理论上不可能，除非beta=1且n=0，但beta<1)
                # 权重是 1 / E_n
                if effective_num < 1e-9: # 如果E_n非常小，权重会非常大，这里可以考虑截断或警告
                                         # 但对于标准的E_n (n>=1, beta<1), E_n >= 1.
                                         # 此处主要防止除以一个极小的数（如果effective_num由于某种原因变得非常小）
                    weights[i] = 1.0 / 1e-9 # 设置一个非常大的权重上限，或者根据策略调整
                else:
                    weights[i] = 1.0 / effective_num
            else:
                # 对于零样本类别，恢复之前的处理方式，赋予权重1.0
                weights[i] = 1.0
                # print(f"   ⚠️ 类别 {i} 样本数为0，权重设为1.0。") # 可选的打印提示

        # # 移除权重限制逻辑
        # if weights.max() > 10.0:
        #     scale_factor = 10.0 / weights.max()
        #     weights = weights * scale_factor
        
        # 归一化权重（可选，默认为False）
        if normalize and num_classes > 0 and np.sum(weights) > 0: # 增加检查避免除以0
            # 只对非零权重进行平均值计算，避免影响零样本的权重（如果它们不是1.0的话）
            # 或者，如果所有权重都可能为0（比如如果零样本也设为0），则需要更小心的处理
            # 在当前情况下，零样本权重为1.0，有样本权重>0，所以np.mean(weights)通常是安全的
            mean_weight = np.mean(weights)
            if mean_weight > 1e-9: # 避免除以0
                 weights = weights / mean_weight
            else:
                 print("   ⚠️ 权重均值过小，跳过归一化。")


        # 返回指定格式
        if return_type == 'numpy':
            return weights
        elif return_type == 'torch':
            return torch.FloatTensor(weights)
        elif return_type == 'dict':
            return {i: float(w) for i, w in enumerate(weights)}
        else:
            raise ValueError(f"不支持的返回类型: {return_type}")


def compute_class_weights_from_labels(labels, beta=0.9999, num_classes=None):
    """
    从标签数组直接计算Effective Number权重
    
    Args:
        labels: array-like, 标签数组
        beta: float, Effective Number的beta参数，默认0.9999
        num_classes: int, 总类别数（可选）
        
    Returns:
        torch.Tensor: 类别权重张量
    """
    # 统计类别分布
    if hasattr(labels, 'numpy'):  # 如果是torch tensor
        labels = labels.numpy()
    
    # 处理one-hot编码
    if len(labels.shape) > 1 and labels.shape[1] > 1:
        labels = np.argmax(labels, axis=1)
    else:
        labels = labels.flatten()
    
    # 统计每个类别的样本数
    class_counts = Counter(labels)
    
    if num_classes is None:
        num_classes = max(class_counts.keys()) + 1
    
    print(f"📊 类别统计: 总样本{len(labels)}, 类别数{num_classes}")
    print(f"📊 样本分布: 最少{min(class_counts.values())}, 最多{max(class_counts.values())}")
    
    # 计算权重
    weights = EffectiveNumberWeights.compute_weights(
        class_counts, beta=beta, num_classes=num_classes, return_type='torch'
    )
    
    print(f"⚖️ 权重范围: [{weights.min():.4f}, {weights.max():.4f}]")
    print(f"🎯 使用beta={beta}")
    
    return weights


def get_effective_weights_for_dataset(dataset, beta=0.9999, label_key='targets'):
    """
    从PyTorch数据集计算Effective Number权重
    
    Args:
        dataset: PyTorch Dataset对象
        beta: float, beta参数
        label_key: str, 标签的属性名（如'targets', 'labels'等）
        
    Returns:
        torch.Tensor: 权重张量
    """
    # 提取所有标签
    if hasattr(dataset, label_key):
        labels = getattr(dataset, label_key)
    elif hasattr(dataset, 'targets'):
        labels = dataset.targets
    elif hasattr(dataset, 'labels'):
        labels = dataset.labels
    else:
        # 遍历数据集获取标签
        labels = []
        for i in range(len(dataset)):
            _, label = dataset[i]
            labels.append(label)
        labels = np.array(labels)
    
    return compute_class_weights_from_labels(labels, beta=beta)


# 使用示例
if __name__ == "__main__":
    print("🔧 Effective Number权重计算工具")
    print("=" * 40)
    
    # 示例1: 从类别统计计算权重
    class_counts = {0: 1000, 1: 500, 2: 100, 3: 50, 4: 10, 5: 5, 6: 1}
    weights = EffectiveNumberWeights.compute_weights(class_counts, beta=0.9999)
    print(f"示例权重: {weights}")
    
    # 示例2: 从标签数组计算权重
    labels = np.random.choice([0, 1, 2], size=1000, p=[0.7, 0.2, 0.1])  # 不平衡标签
    weights = compute_class_weights_from_labels(labels, beta=0.9999)
    print(f"从标签计算的权重: {weights}")
    
    print("\n💡 在训练中使用:")
    print("# 在训练开始前计算权重")
    print("weights = compute_class_weights_from_labels(train_labels, beta=0.9999)")
    print("weights = weights.to(device)")
    print("# 然后在损失函数中使用这个weights")